# 第 2 章 — 分词（Tokenization）

计算机不理解字母，只理解数字。
因此在训练语言模型之前，我们需要把每个词和每个标点符号
都转换成数字。这个过程叫做分词（tokenization），
是现代大语言模型（LLM）的第一步。

本 notebook 将使用与 GPT-2 相同的分词器。
它把词拆成称为子词（subword）的小片段，
从而优雅地处理罕见词和 emoji。
我们会看到 "unbelievably" 如何变成三个 token，
以及 emoji 为何不会出问题。
读完后你将理解文本如何成为 Transformer 模型的原始输入。

## 导入

In [1]:
from dataclasses import dataclass
import tiktoken

## 配置

In [2]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

## 分词器

In [3]:
class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)

    @property
    def vocab_size(self):
        return self.config.vocab_size

## 试一试

In [4]:
tokenizer = SimpleTokenizer()
print(f"词表大小: {tokenizer.vocab_size:,}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print()

text = "The cat sat on the mat."
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)

print(f"原文: '{text}'")
print(f"编码:  {encoded}")
print(f"解码:  '{decoded}'")
print(f"往返一致: {text == decoded}")

Vocabulary size: 50,257
EOS token ID: 50256

Original: 'The cat sat on the mat.'
Encoded:  [464, 3797, 3332, 319, 262, 2603, 13]
Decoded:  'The cat sat on the mat.'
Roundtrip: True


## 罕见词与 emoji

In [5]:
rare = tokenizer.encode("antidisestablishmentarianism")
pieces = [tokenizer.decode([t]) for t in rare]
print(f"子词片段: {pieces}")
print(f"解码: '{tokenizer.decode(rare)}'")
print()

text = tokenizer.encode("Hello world!")
print(f"解码: {tokenizer.decode(text)}")
print()
emoji = tokenizer.encode("Hello \U0001f60a world")
print(f"Emoji 测试: {tokenizer.decode(emoji)}")

Pieces: ['ant', 'idis', 'establishment', 'arian', 'ism']
Decoded: 'antidisestablishmentarianism'

Decoded: Hello world!

Emoji test: Hello 😊 world
